[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/solutions/b_05_prng_keys_solution.ipynb)

# 🟢 Solution: PRNG Keys and Splitting

*JAX Fundamentals · Easy*

Reference implementation. Try it yourself in `b_05_prng_keys.ipynb` first.

---
Initialise an **ensemble** of `n_models` weight matrices from a single PRNG key.

JAX has no global random state. Every draw takes an explicit `key`, and the
same key always produces the same numbers. To get independent randomness you
must **split**.

### Rules
- Give every model its **own** subkey derived from `key` via `jax.random.split`
- Never reuse the same key for two different draws
- Do not consume the caller's `key` directly for the sample itself
- Return shape `(n_models, *shape)`, standard normal values
- `init_ensemble(key, ...)` called twice with the same key must be identical

### Signature
```python
def init_ensemble(key, n_models, shape):  # -> (n_models, *shape)
    ...
```

### The bug this problem exists to teach
```python
# WRONG — every model gets identical weights
[jax.random.normal(key, shape) for _ in range(n)]

# WRONG — subtle: `key` is used for a draw AND carried on to the next split,
# so the two draws below are correlated
key, sub = jax.random.split(key)
a = jax.random.normal(key, shape)     # consumed `key` here...
key, sub = jax.random.split(key)      # ...and split the same `key` again
b = jax.random.normal(key, shape)

# RIGHT — the carry key is only ever split; draws use the subkey
key, sub = jax.random.split(key)
a = jax.random.normal(sub, shape)
```

### Why it matters
Explicit keys are what make JAX reproducible under `jit`, `vmap`, and multi-host
parallelism — the same program gives the same numbers regardless of how it is
compiled or sharded. Every JAX interview probes whether you understand this.

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✅ REFERENCE SOLUTION

import jax
import jax.numpy as jnp


def init_ensemble(key, n_models, shape):
    # One independent subkey per member — never reuse a key across draws.
    keys = jax.random.split(key, n_models)
    return jnp.stack([jax.random.normal(k, shape) for k in keys])

In [ ]:
# 🔍 Verify
import jax

key = jax.random.key(0)
w = init_ensemble(key, 4, (2, 3))
print("shape:", w.shape)
print("member 0 == member 1?", bool((w[0] == w[1]).all()), "(should be False)")

again = init_ensemble(jax.random.key(0), 4, (2, 3))
print("reproducible?", bool((w == again).all()), "(should be True)")

In [ ]:
# Run the judge against the reference solution
from jax_judge import check

check("prng_keys")